# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashkrverma1234-glitch/ml-internship-assignment1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup — connect to the real warehouse (not the starter CSV)
This assignment needs the gated Hugging Face warehouse, queried in place with DuckDB — no full download.
On Colab, store your token as a **Colab Secret** named `HF_TOKEN` (never paste it into a cell). We iterate on the
mid-panel partition `month=2026-03` — never the final month (`2026-06`, sealed test) or `_sample` (also the final
month) — per the panel warning in `skills/flyrank/flyrank-data/SKILL.md`.

In [2]:
%pip -q install duckdb huggingface_hub

import os, getpass

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
DEV_MONTH = "2026-03"  # mid-panel partition -- never 2026-06 or *_sample for label logic

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_month":  f"read_parquet('{REL}/fact_content_daily_performance/month={DEV_MONTH}/*.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:14} {n:>12,} rows")

dim_clients             104 rows
dim_content         519,606 rows
fact_month        9,841,378 rows


## 1. Unit of analysis + time window

**One row = one (client_hash_id, content_hash_id) content item's daily search performance record**, in
`fact_content_daily_performance`. **Time window this notebook: the `month=2026-03` partition only** — one
mid-panel month, picked so I can iterate on query mechanics without touching the sealed final month
(`2026-06`) or the `_sample` table (which *is* that final month). Verified below: schema, row count, and
date span for this exact slice.

In [3]:
# Schema first -- confirm real column names before writing any feature SQL (never guess).
print("fact_content_daily_performance columns:")
display_cols = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_month']}").df()
print(display_cols[["column_name", "column_type"]].to_string(index=False))

print()
print("dim_clients columns:")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_clients']}").df()[["column_name", "column_type"]].to_string(index=False))

# Fact #1 of the "three facts": the grain and window, proven, not asserted.
window = con.sql(f"""
    SELECT COUNT(*) AS rows_in_slice,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM {TABLES['fact_month']}
""").df()
print()
print(window.to_string(index=False))

fact_content_daily_performance columns:
             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemini      BIGIN

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


 rows_in_slice   min_date   max_date  n_clients  n_content_items
       9841378 2026-03-01 2026-03-31         55           331437


## 2. Fields: feature / label / context / excluded

Rest of the plain-words contract:

- **Table(s) I'll use:** `fact_content_daily_performance` (this month's partition, for daily signals)
  and `dim_clients` (for per-client history coverage — `gsc_data_start` / `ga4_data_start` — used in
  Section 4, not as model input). `dim_content` is deliberately **not** joined yet at this dev stage —
  see exclusions.
- **What I'd predict or rank (label or proxy):** at this single-month dev stage there is no real
  forward-looking label yet — one partition can't show "what happens next." Section 3's "trap" uses a
  **within-month demo proxy only** (did impressions fall in the second half of March vs the first) to
  run the leakage experiment honestly. The real capstone label will be a genuine future-window outcome
  (prior N days of features → decline/recovery over the *next* N days), built once this query mechanics
  pass is validated.
- **One thing I deliberately exclude:** any GA4-derived metric (including `sessions_ai`) on rows where
  `ga4_data_available IS NOT TRUE` — zero-filled and NULL rows there are "no tracking yet," not "no
  engagement," so they're excluded rather than trusted as real zeros (verified in Section 3).

Field buckets for this lane, sorted below.

In [4]:
import pandas as pd

field_buckets = pd.DataFrame([
    {"field": "gsc_impressions",       "bucket": "feature", "why": "observable signal, known at month end"},
    {"field": "gsc_clicks",            "bucket": "feature", "why": "observable signal, known at month end"},
    {"field": "gsc_avg_position",      "bucket": "feature", "why": "observable signal; 0 likely means no data -- verify, don't assume"},
    {"field": "sessions_ai",           "bucket": "feature", "why": "observable, but only where ga4_data_available IS TRUE"},
    {"field": "report_date",           "bucket": "context", "why": "defines the window; not a model input itself"},
    {"field": "client_hash_id",        "bucket": "context", "why": "grouping / per-client split key only, never a feature"},
    {"field": "content_hash_id",       "bucket": "context", "why": "grouping / join key only, never a feature"},
    {"field": "ga4_data_available",    "bucket": "context", "why": "gates which rows' GA4 fields are trustworthy"},
    {"field": "is_declining_demo",     "bucket": "label (demo proxy)", "why": "Section 3 only -- within-month proxy for the leakage experiment, not the real capstone label"},
    {"field": "keyword_hash_id / url_hash_id", "bucket": "excluded", "why": "on dim_content, not joined yet at this dev stage; grouping only, never features, when they arrive"},
])
print(field_buckets.to_string(index=False))

                        field             bucket                                                                                               why
              gsc_impressions            feature                                                             observable signal, known at month end
                   gsc_clicks            feature                                                             observable signal, known at month end
             gsc_avg_position            feature                                 observable signal; 0 likely means no data -- verify, don't assume
                  sessions_ai            feature                                             observable, but only where ga4_data_available IS TRUE
                  report_date            context                                                      defines the window; not a model input itself
               client_hash_id            context                                             grouping / per-client spl

## 3. Verify it with queries (grain, counts, missing values, windows)

Three things proven here, plus the five-feature frame and the leakage trap:

1. **Grain check** — group by the claimed grain, `HAVING COUNT(*) > 1` should return nothing.
2. **Availability** — filter `ga4_data_available IS TRUE` and show how many rows actually survive,
   instead of assuming zero-filled rows mean "no engagement."
3. **The five features**, each with a one-line "knowable at the decision moment because…".
4. **The trap** — add one column derived straight from the label, watch the score jump toward
   perfect, then delete it and keep the honest number (the leakage lesson from notebook 02, run here
   on real warehouse data).

In [5]:
# 1) Grain check -- should return zero rows.
dupes = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {TABLES['fact_month']}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"Duplicate (report_date, client_hash_id, content_hash_id) combos: {len(dupes)} (want 0)")

# 2) Availability -- filter with IS TRUE, show what survives.
avail = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)     AS ga4_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_unavailable_or_null_rows
    FROM {TABLES['fact_month']}
""").df()
print()
print(avail.to_string(index=False))
print(f"-> {100 * avail['ga4_available_rows'][0] / avail['total_rows'][0]:.1f}% of this month's rows have usable GA4 data.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (report_date, client_hash_id, content_hash_id) combos: 0 (want 0)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


 total_rows  ga4_available_rows  ga4_unavailable_or_null_rows
    9841378              413966                       9427412
-> 4.2% of this month's rows have usable GA4 data.


In [6]:
# 3) Five features, max -- each knowable before any review decision is made.
monthly = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions)                                                        AS total_impressions,
        SUM(gsc_clicks)                                                             AS total_clicks,
        AVG(NULLIF(gsc_avg_position, 0))                                            AS avg_position,
        COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END)          AS days_with_impressions,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN sessions_ai ELSE 0 END)       AS ai_sessions_total,
        SUM(CASE WHEN report_date <= DATE '{DEV_MONTH}-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
        SUM(CASE WHEN report_date >  DATE '{DEV_MONTH}-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM {TABLES['fact_month']}
    GROUP BY 1, 2
    HAVING total_impressions >= 50
""").df()

print(f"{len(monthly):,} content items with enough March volume for this dev pass")
print()

FEATURE_NOTES = {
    "total_impressions":     "GSC impressions land via the daily sync -- fully known by month end.",
    "total_clicks":          "Same daily GSC sync as impressions -- known by month end.",
    "avg_position":          "Position is measured per query event, aggregated daily -- known by month end.",
    "days_with_impressions": "A count of days already observed in the window -- knowable by construction.",
    "ai_sessions_total":     "GA4-attributed sessions land via the daily analytics sync, but ONLY where "
                              "ga4_data_available IS TRUE -- gated by the availability check above.",
}
for feat, note in FEATURE_NOTES.items():
    print(f"- {feat}: {note}")

monthly.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

116,114 content items with enough March volume for this dev pass

- total_impressions: GSC impressions land via the daily sync -- fully known by month end.
- total_clicks: Same daily GSC sync as impressions -- known by month end.
- avg_position: Position is measured per query event, aggregated daily -- known by month end.
- days_with_impressions: A count of days already observed in the window -- knowable by construction.
- ai_sessions_total: GA4-attributed sessions land via the daily analytics sync, but ONLY where ga4_data_available IS TRUE -- gated by the availability check above.


,client_hash_id,content_hash_id,total_impressions,total_clicks,avg_position,days_with_impressions,ai_sessions_total,imp_first_half,imp_second_half
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,4.888929,24,0.0,57.0,20.0
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,602.0,4.0,4.428747,29,0.0,199.0,403.0
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,810.0,1.0,4.866123,29,0.0,467.0,343.0
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,82.0,0.0,10.100347,27,0.0,56.0,26.0
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,1858.0,6.0,1.854929,30,0.0,771.0,1087.0


In [7]:
# 4) The trap.
# DEMO-ONLY proxy label (Section 2 already flagged this is not the real capstone label):
# did impressions fall in the second half of March vs the first?
monthly["is_declining_demo"] = (monthly["imp_second_half"] < 0.8 * monthly["imp_first_half"]).astype(int)

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ["total_impressions", "total_clicks", "avg_position", "days_with_impressions", "ai_sessions_total"]
model_data = monthly.dropna(subset=honest_features)
X, y = model_data[honest_features], model_data["is_declining_demo"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

honest_model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])
print(f"Honest ROC AUC (5 legit features): {honest_auc:.3f}")

# Now add ONE column derived straight from the label itself.
leaky_features = honest_features + ["imp_second_half"]
X_leak = model_data[leaky_features].copy()
X_leak["imp_second_half"] = monthly.loc[model_data.index, "imp_second_half"]
X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leak, y, test_size=0.25, random_state=42, stratify=y)

leaky_model = LogisticRegression(max_iter=1000).fit(X_tr_l, y_tr_l)
leaky_auc = roc_auc_score(y_te_l, leaky_model.predict_proba(X_te_l)[:, 1])
print(f"Leaky ROC AUC (+ imp_second_half, which the label is LITERALLY built from): {leaky_auc:.3f}")
print("-> that jump is not skill, it is the label leaking straight into the features.")

# Delete the leaky column and keep the honest number.
del X_leak, leaky_features
print()
print(f"Honest number I'm keeping: {honest_auc:.3f} (5 legit features only, no label-derived column)")

Honest ROC AUC (5 legit features): 0.602
Leaky ROC AUC (+ imp_second_half, which the label is LITERALLY built from): 1.000
-> that jump is not skill, it is the label leaking straight into the features.

Honest number I'm keeping: 0.602 (5 legit features only, no label-derived column)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Confirmed limitations, checked against real `dim_clients` data below, not assumed:

- **Unbalanced panel.** Per-client history depth varies — some clients' `gsc_data_start` /
  `ga4_data_start` go back much further than others. Any time window has to be checked per client,
  not assumed globally.
- **GSC-only early history.** Rows before a client's `ga4_data_start` carry zero-filled (or NULL)
  GA4 metrics with `ga4_data_available` not `TRUE` — Section 3 already showed this isn't rare within
  a single month.
- **This one-month slice can't support a real forward-looking label.** `month=2026-03` alone shows
  what happened *during* March, not what happens *after* a decision point — that needs a feature
  window and a separate, later target window spanning multiple partitions, deliberately left for
  later once this month's query mechanics are validated (Section 2).

In [8]:
history = con.sql(f"""
    SELECT
        MIN(gsc_data_start) AS earliest_gsc_start,
        MAX(gsc_data_start) AS latest_gsc_start,
        COUNT(*) FILTER (WHERE ga4_data_start IS NULL) AS clients_with_no_ga4_start,
        COUNT(*) AS n_clients
    FROM {TABLES['dim_clients']}
""").df()
print(history.to_string(index=False))
print()
print("Spread between earliest and latest gsc_data_start is the unbalanced-panel evidence: "
      "a global calendar window would treat clients with barely any history the same as clients "
      "with over a year of it. Per-client windows are the fix, not attempted yet at this dev stage.")

earliest_gsc_start latest_gsc_start  clients_with_no_ga4_start  n_clients
        2025-01-27       2026-06-02                         53        104

Spread between earliest and latest gsc_data_start is the unbalanced-panel evidence: a global calendar window would treat clients with barely any history the same as clients with over a year of it. Per-client windows are the fix, not attempted yet at this dev stage.


## Self-check

Before you submit, confirm each line honestly:

- [yes] Every section above is filled — markdown thinking AND the code that backs it
- [ yes] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ yes] No client names, URLs, or private queries anywhere
- [ yes] My claims use careful words: observed, measured, directional, decision-support
- [ yes] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.